In [ ]:
!pip install -qU langchain langchain-community langchain-groq langchain-text-splitters langchain-huggingface chromadb pypdf sentence-transformers gradio

In [ ]:
import gradio as gr
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import TokenTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"

def process_data(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    splitter = TokenTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    chunks = splitter.split_documents(documents)

    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i
        if "page" not in chunk.metadata:
            chunk.metadata["page"] = "Unknown"

    return chunks


def build_vectorstore(chunks):
    embeddings = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2"
    )

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory="./chroma_db",
        collection_metadata={
            "hnsw:space": "cosine"
        }
    )

    return vectorstore


def ask_gdpr(pdf_file, query):
    if pdf_file is None:
        return (
            "Please upload a GDPR PDF.",
            "No verification available.",
            "No retrieved chunks."
        )

    chunks = process_data(pdf_file.name)
    vectorstore = build_vectorstore(chunks)

    llm = ChatGroq(
        groq_api_key=GROQ_API_KEY,
        model_name="llama-3.3-70b-versatile",
        temperature=0
    )

    docs_and_scores = vectorstore.similarity_search_with_score(
        query,
        k=5
    )

    processed_results = []
    for doc, distance in docs_and_scores:
        cosine_similarity = 1 - distance
        processed_results.append((doc, cosine_similarity))

    processed_results.sort(key=lambda x: x[1], reverse=True)

    SIMILARITY_THRESHOLD = 0.5
    filtered_results = [
        (doc, score)
        for doc, score in processed_results
        if score >= SIMILARITY_THRESHOLD
    ]

    if not filtered_results:
        return (
            "⚠️ No strongly relevant GDPR text found.",
            "WARNING",
            "No retrieved chunks."
        )

    retrieved_docs = [doc for doc, score in filtered_results]
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    # تم استخدام الـ Prompt المحسن من ملف RAG لضمان دقة أعلى
    template = """
You are a GDPR legal assistant.

Answer ONLY using the provided context.

Rules:
- Cite article numbers clearly.
- Only cite article numbers explicitly mentioned in the context.
- Do NOT invent citations.
- If information is missing, say:
  "Not found in retrieved GDPR text."

Context:
{context}

Question:
{question}

Answer:
"""

    prompt = PromptTemplate.from_template(template)
    rag_chain = prompt | llm | StrOutputParser()

    answer = rag_chain.invoke(
        {
            "context": context,
            "question": query
        }
    )

    fact_template = """
You are a strict GDPR fact checker.

Check whether the answer is FULLY supported by the context.

Context:
{context}

Answer:
{answer}

If fully supported reply exactly:
VERIFIED

Otherwise reply exactly:
WARNING
"""

    fact_prompt = PromptTemplate.from_template(fact_template)
    fact_chain = fact_prompt | llm | StrOutputParser()

    verification = fact_chain.invoke(
        {
            "context": context,
            "answer": answer
        }
    )

    chunks_text = ""
    for i, (doc, score) in enumerate(filtered_results):
        chunk_id = doc.metadata.get("chunk_id", "Unknown")
        page_num = doc.metadata.get("page", "Unknown")

        chunks_text += f"""
____________________________
Chunk {i+1}

Chunk ID: {chunk_id}
Page Number: {page_num}
Cosine Similarity: {round(score, 4)}
--------------------------
{doc.page_content}

"""

    return answer, verification, chunks_text


custom_css = """
body {
    background-color: #0f172a;
}
.gradio-container {
    font-family: Arial;
}
.main-title {
    text-align: center;
    font-size: 40px;
    font-weight: bold;
    color: white;
    margin-bottom: 10px;
}
.sub-title {
    text-align: center;
    color: #cbd5e1;
    margin-bottom: 30px;
}
"""

with gr.Blocks() as demo:
    gr.HTML("""
    <div class='main-title'>
        ⚖️ PolicyPulse GDPR Assistant
    </div>
    <div class='sub-title'>
        Retrieval-Augmented Generation for GDPR Legal Analysis
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(
                label="📂 Upload GDPR PDF",
                file_types=[".pdf"]
            )
            question_input = gr.Textbox(
                label="❓ Ask a GDPR Question",
                lines=4,
                placeholder="Example: What does GDPR say about data portability?"
            )
            submit_btn = gr.Button(
                "Generate Answer",
                variant="primary"
            )

        with gr.Column(scale=2):
            answer_output = gr.Textbox(
                label="💡 Final Answer",
                lines=10
            )
            verification_output = gr.Textbox(
                label="🔎 Fact Check Status"
            )

    retrieved_chunks = gr.Accordion(
        "📄 Retrieved Chunks",
        open=False
    )

    with retrieved_chunks:
        chunks_output = gr.Textbox(
            lines=25
        )

    submit_btn.click(
        fn=ask_gdpr,
        inputs=[pdf_input, question_input],
        outputs=[answer_output, verification_output, chunks_output]
    )

demo.launch(share=True, theme=gr.themes.Soft(), css=custom_css)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://128c866d75c8bdcc62.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
